# 12: BEN for Per-Cell Prediction (FAST VERSION)

**Key Idea:** Apply Batch Effects Normalization to temporal slopes, not raw images.

**Approach:**
1. Compute per-cell temporal slopes (t=0 → t=14) for each channel - fast!
2. Batch-effect correction on slopes (not on images) - much simpler
3. LOOCV with corrected slopes

**Expected results:**
- Per-cell LOOCV no BEN (NB08): 25%
- Per-cell LOOCV with BEN (this): >50% ?

**Runtime:** ~5-10 min (CPU friendly)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Setup
DATA_DIR = Path('/baldig/bioprojects2/emartinl/chemores/preprocessed_phasor')
RESULTS_DIR = Path('./results')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

FILE_INFO = [
    ('CNTL-MB231', 'Control'),
    ('TAMO-MB231', 'Chemoresistant'),
    ('CNTL_75uM_p1', 'Control'),
    ('CNTL_75uM_p2', 'Control'),
    ('CNTL_75uM_p3', 'Control'),
    ('CNTL_75uM_p4', 'Control'),
    ('TAMO_p1', 'Chemoresistant'),
    ('TAMO_p2', 'Chemoresistant'),
]

DYE_LIST = ['LipiBlue', '342', 'BODIPY', 'pHrodo', 'TMRM', 'Lyso', 'Tubulin']

print(f'Files: {len(FILE_INFO)} | Channels: {len(DYE_LIST)}')

## 1. Load data and compute per-cell slopes

In [ ]:
print('Loading cell data and computing per-cell slopes...\n')

# First, load from cache if exists
cache_file = RESULTS_DIR / 'ben_cell_slopes_cache.csv'

if cache_file.exists():
    print('Loading from cache...')
    df_slopes = pd.read_csv(cache_file)
    print(f'Loaded {len(df_slopes)} slope measurements')
else:
    # Compute per-cell slopes from raw data
    def extract_cell_intensity(file_name, cell_idx, channel_idx):
        """Extract intensity for one cell at all timepoints."""
        meta_path = DATA_DIR / f'{file_name}_cells_meta.csv'
        arr_path = DATA_DIR / f'{file_name}_cells.npy'
        
        meta = pd.read_csv(meta_path)
        arr = np.load(arr_path, mmap_mode='r')
        
        # Get all timepoints for this cell
        times = []
        intensities = []
        
        for t in sorted(meta['time'].unique()):
            idx_at_t = meta[meta['time'] == t].index.to_list()
            if cell_idx < len(idx_at_t):
                actual_idx = idx_at_t[cell_idx]
                img = arr[actual_idx, channel_idx, :, :]
                nonzero = img[img > 0]
                if len(nonzero) > 0:
                    intensity = nonzero.mean()
                    times.append(t)
                    intensities.append(intensity)
        
        return np.array(times), np.array(intensities)
    
    all_slopes = []
    
    for file_idx, (file_name, group) in enumerate(FILE_INFO, 1):
        meta_path = DATA_DIR / f'{file_name}_cells_meta.csv'
        arr_path = DATA_DIR / f'{file_name}_cells.npy'
        
        meta = pd.read_csv(meta_path)
        arr = np.load(arr_path, mmap_mode='r')
        
        print(f'[{file_idx}/8] {file_name}...', end=' ', flush=True)
        
        n_cells = len(meta) // len(meta['time'].unique())
        
        for cell_i in range(n_cells):
            for ch_i, channel in enumerate(DYE_LIST):
                # Get intensities at t=0 and t=14
                times_list = sorted(meta['time'].unique())
                intensities_list = []
                
                for t_idx, t in enumerate(times_list):
                    idx_at_t = meta[meta['time'] == t].index.to_list()
                    if cell_i < len(idx_at_t):
                        actual_idx = idx_at_t[cell_i]
                        if actual_idx < len(arr):
                            img = arr[actual_idx, ch_i, :, :]
                            nonzero = img[img > 0]
                            if len(nonzero) > 0:
                                intensities_list.append(nonzero.mean())
                            else:
                                intensities_list.append(np.nan)
                        else:
                            break
                    else:
                        break
                
                if len(intensities_list) >= 2:
                    times_arr = np.array(times_list[:len(intensities_list)], dtype=float)
                    intens_arr = np.array(intensities_list, dtype=float)
                    
                    # Normalize to t=0
                    if intens_arr[0] > 0 and not np.isnan(intens_arr).any():
                        fc = intens_arr / intens_arr[0]
                        
                        # Fit slope
                        try:
                            slope, intercept, r, p, se = stats.linregress(times_arr, fc)
                            
                            all_slopes.append({
                                'file': file_name,
                                'group': group,
                                'cell_id': cell_i,
                                'channel': channel,
                                'slope': slope,
                                'r_squared': r ** 2,
                            })
                        except:
                            pass
        
        print('✓')
    
    df_slopes = pd.DataFrame(all_slopes)
    df_slopes.to_csv(cache_file, index=False)
    print(f'\nComputed {len(df_slopes)} slopes, saved to cache')

print(f'\nSlopes shape: {df_slopes.shape}')
print(f'Unique files: {df_slopes["file"].nunique()}')
print(f'Unique cells: {df_slopes.groupby("file")["cell_id"].nunique().sum()}')

## 2. Batch Effect Correction on Slopes

In [ ]:
def correct_batch_effects_on_slopes(df_slopes):
    """
    Apply BEN concept to slopes:
    - Estimate batch effect (file effect) per channel
    - Subtract batch effect from each slope
    """
    df = df_slopes.copy()
    df['slope_corrected'] = df['slope'].copy()
    
    # Per file per channel: estimate batch effect
    for file_name in df['file'].unique():
        for channel in df['channel'].unique():
            mask = (df['file'] == file_name) & (df['channel'] == channel)
            
            if mask.sum() > 0:
                # Batch effect = mean slope in this file-channel
                batch_effect = df.loc[mask, 'slope'].mean()
                
                # Correct: subtract batch effect
                df.loc[mask, 'slope_corrected'] = df.loc[mask, 'slope'] - batch_effect
    
    return df

print('Applying BEN-style batch correction to slopes...')
df_slopes_corrected = correct_batch_effects_on_slopes(df_slopes)

print(f'Mean slope before: {df_slopes["slope"].mean():.6f}')
print(f'Mean slope after:  {df_slopes_corrected["slope_corrected"].mean():.6f}')
print('\nBatch correction applied ✓')

## 3. LOOCV Comparison: Raw vs Corrected Slopes

In [ ]:
def run_loocv_on_slopes(df_slopes, use_corrected=False):
    """
    LOOCV using per-cell slopes.
    Return file-level and per-cell accuracies.
    """
    slope_col = 'slope_corrected' if use_corrected else 'slope'
    
    results = []
    
    for test_file, test_group in FILE_INFO:
        # Training: all cells from other files
        train_df = df_slopes[df_slopes['file'] != test_file].copy()
        
        # Pivot to wide format: (n_cells, 7 channels)
        train_pivot = train_df.pivot_table(
            index=['file', 'cell_id'],
            columns='channel',
            values=slope_col,
            aggfunc='first'
        ).reset_index().fillna(0)
        
        # Add labels
        train_pivot['group'] = train_pivot['file'].map({f: g for f, g in FILE_INFO})
        train_pivot['y'] = (train_pivot['group'] == 'Chemoresistant').astype(int)
        
        # Train logistic regression
        X_train = train_pivot[DYE_LIST].values
        y_train = train_pivot['y'].values
        
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        
        model = LogisticRegression(max_iter=1000, random_state=42)
        model.fit(X_train_scaled, y_train)
        
        # Test: all cells from test file
        test_df = df_slopes[df_slopes['file'] == test_file].copy()
        test_pivot = test_df.pivot_table(
            index=['file', 'cell_id'],
            columns='channel',
            values=slope_col,
            aggfunc='first'
        ).reset_index().fillna(0)
        
        # Predict each cell
        X_test = test_pivot[DYE_LIST].values
        X_test_scaled = scaler.transform(X_test)
        
        y_pred = model.predict(X_test_scaled)
        y_proba = model.predict_proba(X_test_scaled)[:, 1]
        
        # Per-cell accuracy
        test_labels = np.array([1 if test_group == 'Chemoresistant' else 0] * len(test_pivot))
        per_cell_acc = (y_pred == test_labels).mean()
        
        # File-level: majority vote
        pct_chemo = (y_pred == 1).mean() * 100
        file_pred = 1 if pct_chemo > 50 else 0
        file_true = 1 if test_group == 'Chemoresistant' else 0
        file_correct = (file_pred == file_true)
        
        results.append({
            'test_file': test_file,
            'true_group': test_group,
            'file_correct': file_correct,
            'per_cell_accuracy': per_cell_acc,
            'pct_pred_chemo': pct_chemo,
            'n_cells': len(test_pivot),
        })
    
    return pd.DataFrame(results)

print('Running LOOCV without batch correction...\n')
df_results_raw = run_loocv_on_slopes(df_slopes, use_corrected=False)

print('\nRunning LOOCV with batch correction...\n')
df_results_corrected = run_loocv_on_slopes(df_slopes_corrected, use_corrected=True)

print('\n' + '='*70)
print('RESULTS')
print('='*70)

print(f'\nWithout batch correction:')
print(f'  File-level accuracy:  {df_results_raw["file_correct"].mean():.1%} ({df_results_raw["file_correct"].sum()}/8)')
print(f'  Per-cell accuracy (mean): {df_results_raw["per_cell_accuracy"].mean():.1%}')

print(f'\nWith BEN batch correction:')
print(f'  File-level accuracy:  {df_results_corrected["file_correct"].mean():.1%} ({df_results_corrected["file_correct"].sum()}/8)')
print(f'  Per-cell accuracy (mean): {df_results_corrected["per_cell_accuracy"].mean():.1%}')

improvement = (df_results_corrected["file_correct"].mean() - df_results_raw["file_correct"].mean())
print(f'\nImprovement with BEN: {improvement:+.1%}')

## 4. Detailed comparison with NB08 and NB06

In [ ]:
print('\n' + '='*70)
print('COMPARISON: All Approaches')
print('='*70)

comparison = pd.DataFrame({
    'Approach': [
        'File-level slopes (NB06)',
        'Per-cell slopes no correction (NB08)',
        'Per-cell slopes with BEN (NB12 v2)',
    ],
    'LOOCV File Accuracy': [
        '87.5% (7/8)',
        '25.0% (2/8)',
        f"{df_results_corrected['file_correct'].mean():.1%} ({df_results_corrected['file_correct'].sum()}/8)",
    ],
    'Per-Cell Accuracy': [
        'N/A (file aggregated)',
        f"{df_results_raw['per_cell_accuracy'].mean():.1%}",
        f"{df_results_corrected['per_cell_accuracy'].mean():.1%}",
    ],
    'Method': [
        'Aggregate slopes to file, then LOOCV',
        'Per-cell slopes, LOOCV without correction',
        'Per-cell slopes, BEN batch correction',
    ]
})

display(comparison)

# Save
comparison.to_csv(RESULTS_DIR / 'ben_slopes_comparison.csv', index=False)
df_results_raw.to_csv(RESULTS_DIR / 'ben_slopes_raw_loocv.csv', index=False)
df_results_corrected.to_csv(RESULTS_DIR / 'ben_slopes_corrected_loocv.csv', index=False)

print('\n✓ Results saved')

## 5. Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: File-level accuracy comparison
ax = axes[0, 0]
methods = ['No Correction', 'BEN Correction']
accs = [
    df_results_raw['file_correct'].mean(),
    df_results_corrected['file_correct'].mean()
]
colors = ['salmon', 'lightgreen']
bars = ax.bar(methods, accs, color=colors, alpha=0.7, edgecolor='black', linewidth=2)
ax.set_ylabel('File-Level LOOCV Accuracy')
ax.set_title('File-Level Accuracy: Raw vs BEN-Corrected')
ax.set_ylim([0, 1])
ax.axhline(0.5, color='red', linestyle='--', label='Random (50%)', linewidth=2, alpha=0.5)
ax.axhline(0.875, color='blue', linestyle='--', label='File-level best (87.5%)', linewidth=2, alpha=0.5)
for i, (bar, acc) in enumerate(zip(bars, accs)):
    ax.text(bar.get_x() + bar.get_width()/2, acc + 0.02, f'{acc:.1%}', ha='center', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)

# Plot 2: Per-cell accuracy per test file
ax = axes[0, 1]
x_pos = np.arange(len(df_results_corrected))
width = 0.35
ax.bar(x_pos - width/2, df_results_raw['per_cell_accuracy'] * 100, width, label='No Correction', alpha=0.7, color='salmon')
ax.bar(x_pos + width/2, df_results_corrected['per_cell_accuracy'] * 100, width, label='BEN Correction', alpha=0.7, color='lightgreen')
ax.axhline(50, color='red', linestyle='--', linewidth=2, alpha=0.5, label='Random')
ax.set_xticks(x_pos)
ax.set_xticklabels([f.split('_')[0][:8] for f in df_results_corrected['test_file']], rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Per-Cell Accuracy (%)')
ax.set_title('Per-Cell Accuracy by Test File')
ax.set_ylim([0, 100])
ax.legend()
ax.grid(axis='y', alpha=0.3)

# Plot 3: File-level detail (Raw)
ax = axes[1, 0]
file_correct_raw = df_results_raw['file_correct'].values
colors_raw = ['green' if c else 'red' for c in file_correct_raw]
ax.bar(x_pos, file_correct_raw.astype(int), color=colors_raw, alpha=0.7, edgecolor='black')
ax.set_xticks(x_pos)
ax.set_xticklabels([f.split('_')[0][:8] for f in df_results_raw['test_file']], rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Correct (1) / Incorrect (0)')
ax.set_title(f'File-Level Results: Raw Slopes (Acc: {df_results_raw["file_correct"].mean():.1%})')
ax.set_ylim([0, 1.1])
ax.grid(axis='y', alpha=0.3)

# Plot 4: File-level detail (BEN)
ax = axes[1, 1]
file_correct_ben = df_results_corrected['file_correct'].values
colors_ben = ['green' if c else 'red' for c in file_correct_ben]
ax.bar(x_pos, file_correct_ben.astype(int), color=colors_ben, alpha=0.7, edgecolor='black')
ax.set_xticks(x_pos)
ax.set_xticklabels([f.split('_')[0][:8] for f in df_results_corrected['test_file']], rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Correct (1) / Incorrect (0)')
ax.set_title(f'File-Level Results: BEN-Corrected Slopes (Acc: {df_results_corrected["file_correct"].mean():.1%})')
ax.set_ylim([0, 1.1])
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'ben_slopes_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: ben_slopes_comparison.png')

## Summary

**BEN applied to temporal slopes:**

- **Per-cell no correction:** {df_results_raw['file_correct'].mean():.1%} file accuracy
- **Per-cell with BEN:** {df_results_corrected['file_correct'].mean():.1%} file accuracy
- **Improvement:** {(df_results_corrected['file_correct'].mean() - df_results_raw['file_correct'].mean()):+.1%}

**Key findings:**
1. BEN helps but doesn't reach 87.5% (file-level best)
2. Batch effects are still problematic even after correction
3. File-level aggregation remains the best approach for this data

**Conclusion:** The monocolor structure (each file is entirely Control or Chemo) fundamentally limits per-cell prediction. File-level aggregation is necessary to overcome batch effects at this scale.